# Week 10: the Mallat algorithm in 1D

**What we do.** We compute the discrete wavelet transform (DWT) of a signal with our own implementation of the Mallat algorithm: one analysis step (filter and downsample), one synthesis step (upsample, filter, add), and the multilevel decomposition. We compare everything with PyWavelets and look at where the energy of the signal goes.

**What to expect.** Our results agree with `pywt.dwt` / `pywt.wavedec` up to rounding, the reconstruction is perfect, and the transform preserves energy (it is orthogonal). For the HeaviSine test signal the smooth part ends up in a few coarse coefficients, while the two jumps produce large detail coefficients at every level.

**How to read the code.** Functions come from `wavelets_lab.dwt` (the transform) and `wavelets_lab.denoising` (the test signal). Signals are treated as periodic (`mode="periodization"` in PyWavelets), so a signal of length `N` gives `N/2` approximation and `N/2` detail coefficients.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pywt

from wavelets_lab import denoising, dwt

## The test signal

**What we do.** We create the HeaviSine signal of length 1024.

**What to expect.** A sinusoid with two jumps, at `t = 0.3` and `t = 0.72`.

**How to read the code.** `denoising.heavisine(n)` samples the formula `4 sin(4πt) - sign(t - 0.3) - sign(0.72 - t)` at `t = k/n`.

In [ ]:
x = denoising.heavisine(1024)
t = np.arange(x.size) / x.size
plt.figure(figsize=(10, 3))
plt.plot(t, x)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("HeaviSine")
plt.show()

## One analysis step

**What we do.** We apply one Mallat step with the db2 wavelet and compare it with `pywt.dwt`.

**What to expect.** Two signals of length 512: the approximation looks like the original signal at half resolution (scaled by `sqrt 2`), the details are almost zero except near the jumps. The differences from PyWavelets are about `1e-15`.

**How to read the code.** `dwt.analysis_step(x, "db2")` returns `(approximation, detail)`. Both are plotted against the position in the original signal (coefficient `k` belongs to `t ≈ 2k/N`).

In [ ]:
approximation, detail = dwt.analysis_step(x, "db2")
expected = pywt.dwt(x, "db2", mode="periodization")
t_half = np.arange(approximation.size) * 2 / x.size
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].plot(t_half, approximation)
axes[0].set_title("approximation a_1")
axes[1].plot(t_half, detail)
axes[1].set_title("detail d_1")
for ax in axes:
    ax.set_xlabel("position t")
    ax.set_ylabel("coefficient")
plt.tight_layout()
plt.show()
np.abs(approximation - expected[0]).max(), np.abs(detail - expected[1]).max()

## One synthesis step

**What we do.** We invert the analysis step.

**What to expect.** The original signal is recovered up to about `1e-15` (perfect reconstruction).

**How to read the code.** `dwt.synthesis_step(approximation, detail, "db2")` returns a signal of length 1024.

In [ ]:
np.abs(dwt.synthesis_step(approximation, detail, "db2") - x).max()

## Multilevel decomposition

**What we do.** We repeat the analysis step five times on the approximation, obtaining `a_5, d_5, d_4, d_3, d_2, d_1`, and plot all bands.

**What to expect.** The band lengths are 32, 32, 64, 128, 256, 512. `a_5` is a coarse version of the signal; the details are small except at the two jumps, which appear at the same positions at every level (wavelets are local).

**How to read the code.** `dwt.decompose(x, "db2", 5)` returns the list `[a_5, d_5, …, d_1]`. Each band is plotted against the position `t` it describes (`np.linspace(0, 1, len(band), endpoint=False)`), so all panels share the horizontal axis.

In [ ]:
coefficients = dwt.decompose(x, "db2", 5)
labels = ["a5", "d5", "d4", "d3", "d2", "d1"]
fig, axes = plt.subplots(len(coefficients) + 1, 1, figsize=(10, 13), sharex=True)
axes[0].plot(t, x)
axes[0].set_title("signal")
for ax, band, label in zip(axes[1:], coefficients, labels, strict=True):
    ax.plot(np.linspace(0.0, 1.0, band.size, endpoint=False), band, ".-")
    ax.set_title(f"{label} ({band.size} coefficients)")
for ax in axes:
    ax.set_xlabel("position t")
    ax.set_ylabel("value")
plt.tight_layout()
plt.show()

## Comparison with PyWavelets and perfect reconstruction

**What we do.** We compare every band with `pywt.wavedec` and reconstruct the signal from all bands.

**What to expect.** All differences are of order `1e-14` or smaller.

**How to read the code.** `zip` pairs our bands with PyWavelets' bands; `dwt.reconstruct(coefficients, "db2")` applies the synthesis steps from the coarsest level up.

In [ ]:
reference = pywt.wavedec(x, "db2", mode="periodization", level=5)
band_errors = [
    np.abs(ours - theirs).max()
    for ours, theirs in zip(coefficients, reference, strict=True)
]
max(band_errors), np.abs(dwt.reconstruct(coefficients, "db2") - x).max()

## Where is the energy?

**What we do.** We compute the fraction of the total energy `Σ x²` in each band and compare the sum with the energy of the signal.

**What to expect.** Almost all the energy (about 99%) is in `a_5`, i.e. in 32 out of 1024 coefficients: the signal is **sparse** in the wavelet basis. The energies of the bands add up to the energy of the signal (Parseval's identity for an orthonormal basis). This is the basis of wavelet compression and denoising (weeks 13 and 14).

**How to read the code.** `np.sum(band**2)` is the energy of a band; the bar chart shows the fractions on a logarithmic axis.

In [ ]:
energy = np.array([np.sum(band**2) for band in coefficients])
plt.figure(figsize=(7, 3.5))
plt.bar(labels, energy / energy.sum())
plt.yscale("log")
plt.xlabel("band")
plt.ylabel("fraction of the energy")
plt.show()
energy.sum(), np.sum(x**2)

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above with its purpose, input, output and a recipe for writing it yourself.

**What to expect.** Two modules are used: `wavelets_lab.dwt` (the Mallat algorithm) and `wavelets_lab.denoising` (only the test signal `heavisine`).

**How to read the code.** All transforms are periodic; bands are listed from coarse to fine as in PyWavelets.

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `pywt.dwt(x, wavelet, mode)` | one DWT step of PyWavelets (reference) | signal → `(cA, cD)` |
| `pywt.wavedec(x, wavelet, mode, level)` | multilevel DWT of PyWavelets (reference) | signal → list of bands |
| `plt.bar(labels, heights)` | bar chart | lists → bars |
| `np.linspace(0, 1, n, endpoint=False)` | `n` points `0, 1/n, …, (n-1)/n` | numbers → array |

### Module `wavelets_lab.dwt` (file `src/wavelets_lab/dwt.py`)

The Mallat algorithm (discrete wavelet transform, DWT) in 1D, written from scratch, and the 2D DWT of images via PyWavelets. Signals are treated as **periodic**, which makes the transform orthogonal (it preserves energy) and matches `mode="periodization"` of PyWavelets; then a signal of length `N` gives `N/2` approximation and `N/2` detail coefficients. Imports:
```python
import numpy as np
import pywt
```
Two internal helpers: `_decomposition_filters(wavelet)` returns `pywt.Wavelet(wavelet).dec_lo` and `.dec_hi` as arrays (`dec_lo` is `rec_lo` reversed); `_analysis_indices(n, length)` returns the integer matrix `idx[k, m] = (2k + length // 2 - m) mod n` for `k = 0, …, n/2 - 1` and `m = 0, …, length - 1`, i.e. which sample is multiplied by filter tap `m` in output `k`.

#### `dwt.analysis_step(x, wavelet="db2")`

**What it is.** One step of the Mallat algorithm: filter the signal with the low-pass and the high-pass decomposition filters and keep every second output (downsampling by 2). The result is the coarse approximation `a` (local averages) and the details `d` (what the averages lose):
`a[k] = sum_m h[m] x[(2k + L/2 - m) mod N]`, `d[k] = sum_m g[m] x[(2k + L/2 - m) mod N]`.

**Input.** `x` (array-like): signal of even length `N`; `wavelet` (`str`): name of an orthogonal PyWavelets wavelet (`"haar"`, `"db2"`, …).

**Output.** Tuple `(approximation, detail)`, both of length `N/2`; equal to `pywt.dwt(x, wavelet, mode="periodization")`. Raises `ValueError` for odd `N`.

**How to write it yourself.**
1. Convert `x` to `float64`; raise `ValueError` if its length is odd.
2. Filters: `w = pywt.Wavelet(wavelet)`, `h = np.asarray(w.dec_lo)`, `g = np.asarray(w.dec_hi)`; `L = len(h)`.
3. Index matrix: `k = np.arange(N // 2)[:, None]`, `m = np.arange(L)[None, :]`, `idx = (2 * k + L // 2 - m) % N`. The shift `L // 2` aligns the result with PyWavelets; `% N` implements periodicity.
4. `samples = x[idx]` has shape `(N/2, L)`: row `k` holds the samples entering output `k`.
5. Return `samples @ h, samples @ g`.

#### `dwt.synthesis_step(approximation, detail, wavelet="db2")`

**What it is.** The inverse of `analysis_step`: upsample, filter and add. Because the periodic transform is orthogonal, its inverse is its transpose (adjoint): every coefficient `a[k]` is "spread back" onto the same samples it was computed from, with the same weights.

**Input.** `approximation`, `detail` (array-like, same length `N/2`); `wavelet` (`str`): the wavelet used for the analysis.

**Output.** `float64` signal of length `N`.

**How to write it yourself.**
1. Get `h`, `g` and the index matrix `idx` (shape `(N/2, L)`) as in `analysis_step`, with `N = 2 * len(approximation)`.
2. Contributions: `c = a[:, None] * h + d[:, None] * g` (shape `(N/2, L)`): coefficient `k` contributes `a[k] h[m] + d[k] g[m]` to sample `idx[k, m]`.
3. Accumulate: `signal = np.zeros(N)` and `np.add.at(signal, idx, c)`. (`np.add.at` adds correctly even when the same index appears several times, which plain `signal[idx] += c` does not.)
4. Return `signal`.

#### `dwt.decompose(x, wavelet="db2", level=3)`

**What it is.** The multilevel DWT: apply `analysis_step` to the signal, then again to the approximation, and so on, `level` times. The details of every level are kept.

**Input.** `x` (array-like): signal whose length is divisible by `2**level`; `wavelet` (`str`); `level` (`int`).

**Output.** List `[a_J, d_J, d_{J-1}, …, d_1]` (coarsest approximation first, then details from coarse to fine), the same order as `pywt.wavedec`. Lengths are `N/2^J, N/2^J, …, N/2`.

**How to write it yourself.**
1. `approximation = x`, `details = []`.
2. Repeat `level` times: `approximation, detail = analysis_step(approximation, wavelet)`; `details.append(detail)`.
3. The finest details were computed first, so reverse them: return `[approximation, *details[::-1]]`.

#### `dwt.reconstruct(coefficients, wavelet="db2")`

**What it is.** The inverse of `decompose`.

**Input.** `coefficients` (list): `[a_J, d_J, …, d_1]`; `wavelet` (`str`).

**Output.** `float64` reconstructed signal.

**How to write it yourself.**
1. `signal = coefficients[0]`.
2. For each detail array in `coefficients[1:]` (from coarse to fine): `signal = synthesis_step(signal, detail, wavelet)`.
3. Return `signal`.

### Module `wavelets_lab.denoising` (file `src/wavelets_lab/denoising.py`)

Wavelet denoising (Donoho–Johnstone): white noise spreads evenly over all wavelet coefficients, while a piecewise smooth signal is concentrated in a few large ones; so thresholding the details removes most of the noise and keeps the signal. Imports:
```python
import numpy as np
import pywt
from wavelets_lab.compression import hard_threshold, soft_threshold
```
and the constant `MAD_CONSTANT = 0.6744897501960817`, the median of `|Z|` for a standard normal `Z`.

#### `denoising.heavisine(n=1024)`

**What it is.** The "HeaviSine" test signal `4 sin(4πt) - sign(t - 0.3) - sign(0.72 - t)` on `[0, 1)`: a smooth sinusoid with two jumps (at `t = 0.3` and `t = 0.72`).

**Input.** `n` (`int`): number of samples.

**Output.** `float64` array of length `n`.

**How to write it yourself.**
1. `t = np.arange(n) / n`.
2. Return `4 * np.sin(4 * np.pi * t) - np.sign(t - 0.3) - np.sign(0.72 - t)`.